# 계류장 장비까지 찾는 YOLO 추가 학습 (Colab GPU)

공개 계류장 데이터셋(Roboflow Universe, CC BY 4.0)으로 **YOLO26x**를 추가 학습해서
비행기·사람·차량에 더해 **수하물 트럭, 벨트 로더, 급유차, 토잉카, 계단차, 탑승교, 케이터링 트럭, 지상전원장치** 등을 찾게 합니다.

**실행 전**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`(또는 더 좋은 GPU)를 고릅니다.
2. Roboflow 무료 계정을 만들고 `Settings → API Keys`에서 Private API Key를 복사해 둡니다. 키는 아래 칸에서 입력창으로만 받고 노트북에 저장하지 않습니다.
3. `런타임 → 모두 실행`. T4 기준 몇 시간 걸립니다. 중간에 끊기면 마지막 칸의 '이어서 학습'을 실행합니다.

**데이터** (2026-11-18 기준)

| 데이터셋 | 이미지 | 주요 물체 |
|---|---|---|
| [Handling](https://universe.roboflow.com/datalabeling-0nprc/handling) | 300 | 비행기, 수하물 트럭, 벨트 로더, 급유차, 토잉카, 계단차, 탑승교, 케이터링, GPU 등 19종 |
| [airport_90](https://universe.roboflow.com/airport-azgu2/airport_90) | 90 | 승용차, 버스, 트럭, 승객, 직원, 탑승교, 토잉카 등 12종 |
| [combine_datasets](https://universe.roboflow.com/aircraftturnaround/combine_datasets) (선택) | 1,034 | 사람, 버스, 급유차, 계단차, GPU 등 9종. 받을 수 있는 판이 없어서, 웹에서 **Fork Dataset → Generate Version**을 한 뒤 아래 `EXTRA`에 내 작업공간 이름을 넣어야 합니다 |

데이터가 작아서(약 400~1,400장) 장비 종류를 새로 배우는 데 의미가 있고, 사람·승용차는 원래 COCO 모델이 더 잘 찾을 수 있습니다. 마지막에 테스트 구간 성적을 종류별로 확인합니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.215 roboflow

## 1. 데이터 받기

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Roboflow Private API Key: "))

SOURCES = [("datalabeling-0nprc", "handling"), ("airport-azgu2", "airport_90")]
# combine_datasets 를 내 작업공간으로 Fork + Generate Version 했다면 여기에 (내 작업공간, 프로젝트 이름) 을 넣는다
EXTRA = []   # 예: [("my-workspace", "combine_datasets")]

paths = []
for ws, name in SOURCES + EXTRA:
    proj = rf.workspace(ws).project(name)
    latest = max(int(str(v.version).split("/")[-1]) for v in proj.versions())
    ds = proj.version(latest).download("yolov8", location=f"/content/raw/{name}")
    paths.append(ds.location)
    print(name, "판", latest, "->", ds.location)

## 2. 종류 이름 맞추기, 합치기

데이터셋마다 이름이 달라서 하나의 목록으로 묶습니다. 문, 엔진, 호스처럼 혼잡도와 상관없는 종류는 뺍니다.
각 데이터셋의 train/valid/test 나눔은 그대로 둡니다. 모델 고르기는 valid 성적으로만 하고, test는 마지막 확인에만 씁니다.

In [ ]:
import shutil
from pathlib import Path

import yaml

CLASSES = ["aircraft", "person", "car", "bus", "truck", "baggage_truck", "belt_loader", "fuel_truck",
           "tug", "stairs", "jet_bridge", "catering_truck", "container_loader", "gpu"]
MAP = {  # 원래 이름(소문자) -> 새 이름. 없는 이름은 버린다
    "plane": "aircraft", "aircraft": "aircraft", "airplane": "aircraft",
    "person": "person", "passenger": "person", "staff": "person",
    "car": "car", "bus": "bus", "passenger bus": "bus", "truck": "truck", "fire truck": "truck", "water service": "truck",
    "baggage truck": "baggage_truck", "baggage_truck": "baggage_truck", "container transport": "baggage_truck",
    "belt loader": "belt_loader", "rump_loader": "belt_loader",
    "fuel truck": "fuel_truck", "fuel_truck": "fuel_truck",
    "pushback truck": "tug", "aircraft tractor": "tug",
    "boarding stairway": "stairs", "aircraft ladder": "stairs", "rolling_stairway": "stairs", "stairway": "stairs",
    "jet bridge": "jet_bridge", "boarding bridge": "jet_bridge",
    "catering truck": "catering_truck", "lift truck": "catering_truck",
    "container loader": "container_loader", "gpu": "gpu", "ground_power": "gpu",
}
OUT = Path("/content/apron")
shutil.rmtree(OUT, ignore_errors=True)
count = {c: 0 for c in CLASSES}
for root in map(Path, paths):
    names = yaml.safe_load((root / "data.yaml").read_text())["names"]
    names = list(names.values()) if isinstance(names, dict) else names
    remap = {i: CLASSES.index(MAP[n.lower().strip()]) for i, n in enumerate(names) if n.lower().strip() in MAP}
    print(root.name, "버리는 종류:", [n for n in names if n.lower().strip() not in MAP])
    for split in ("train", "valid", "test"):
        img_dir = root / split / "images"
        if not img_dir.exists():
            continue
        (OUT / split / "images").mkdir(parents=True, exist_ok=True)
        (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
        for img in img_dir.iterdir():
            lab = root / split / "labels" / (img.stem + ".txt")
            lines = []
            if lab.exists():
                for line in lab.read_text().splitlines():
                    p = line.split()
                    if p and int(p[0]) in remap:
                        lines.append(" ".join([str(remap[int(p[0])])] + p[1:]))
                        count[CLASSES[remap[int(p[0])]]] += split == "train"
            new = f"{root.name}_{img.name}"
            shutil.copy(img, OUT / split / "images" / new)
            (OUT / split / "labels" / (Path(new).stem + ".txt")).write_text("\n".join(lines))

(OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
                                               "test": "test/images", "names": dict(enumerate(CLASSES))}))
for split in ("train", "valid", "test"):
    print(split, len(list((OUT / split / "images").glob("*"))), "장")
print("학습 구간 박스 수:", count)

## 3. 학습

- 시작점: YOLO26x (COCO로 미리 학습된 가장 큰 모델)
- 사진이 적어서 늘리기(회전·크기·색 변화, 모자이크)를 켜고, valid 성적이 30번 연속 안 좋아지면 멈춥니다.
- 결과는 `/content/runs/apron/yolo26x`에 쌓입니다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26x.pt")
model.train(data="/content/apron/data.yaml", epochs=200, patience=30, imgsz=1024, batch=-1,
            cos_lr=True, close_mosaic=15, degrees=5, scale=0.5, fliplr=0.5, hsv_v=0.4,
            project="/content/runs/apron", name="yolo26x", exist_ok=True, seed=0, deterministic=True, plots=True)

## 4. 테스트 구간 성적 (마지막에 한 번만)

In [ ]:
best = YOLO("/content/runs/apron/yolo26x/weights/best.pt")
m = best.val(data="/content/apron/data.yaml", split="test", imgsz=1024, plots=True)
import pandas as pd
rows = [{"종류": best.names[c], "mAP50": round(float(m.box.ap50[i]), 3), "mAP50-95": round(float(m.box.ap[i]), 3)}
        for i, c in enumerate(m.box.ap_class_index)]
print(pd.DataFrame(rows).to_string(index=False))
print("전체 mAP50:", round(m.box.map50, 3), " mAP50-95:", round(m.box.map, 3))

## 5. 가중치 내려받기

`apron_yolo26x.pt`와 학습 기록(results.csv, 그래프)을 받아서 저장소의 `ojh/models/saved/yolo/`에 넣습니다.
이 폴더는 git에 올라가지 않습니다(용량). 화면 연결은 받은 뒤에 합니다.

In [ ]:
import shutil
from google.colab import files

shutil.copy("/content/runs/apron/yolo26x/weights/best.pt", "/content/apron_yolo26x.pt")
shutil.make_archive("/content/apron_train_log", "zip", "/content/runs/apron/yolo26x")
files.download("/content/apron_yolo26x.pt")
files.download("/content/apron_train_log.zip")

## (끊겼을 때) 이어서 학습

In [ ]:
# YOLO("/content/runs/apron/yolo26x/weights/last.pt").train(resume=True)